# Guardian / BBC / Daily Mail — Brexit & Covid Datasets (fast version)




In [1]:
!pip install -q mediacloud trafilatura requests pandas tqdm python-dateutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/brexit_covid_datasets"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Results will be saved to: {OUTPUT_DIR}")

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/brexit_covid_datasets


In [7]:
import pandas as pd
from datetime import datetime
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from getpass import getpass

import mediacloud.api
import trafilatura
import requests

# ---- CONFIG ----
MC_API_KEY = getpass("Paste your Media Cloud API key: ")
GUARDIAN_API_KEY = getpass("Paste your Guardian Open Platform API key: ")

MC_OUTLETS = {
    "bbc": "bbc.co.uk",
    "dailymail": "dailymail.co.uk",
}

# If automatic domain-name search (below, in resolve_source_id) can't find a match,
# look the source up manually at https://search.mediacloud.org/directory
# (search "BBC" / "Daily Mail"), open it, and paste its numeric id here.
# Leave as None to keep relying on the automatic search.
MANUAL_SOURCE_IDS = {
    "bbc": None,
    "dailymail": 19142,
}

BREXIT_KEYWORDS = [
    "Brexit", "EU referendum", "leave the EU", "Article 50",
    "Brexit deal", "no-deal Brexit", "Brexit negotiations", "Brexit vote",
]

COVID_KEYWORDS = [
    "coronavirus", "COVID-19", "COVID", "pandemic UK",
    "lockdown UK", "NHS coronavirus", "vaccine rollout UK", "Covid restrictions",
]

END_DATE = datetime.now().strftime("%Y-%m-%d")

# topic-specific start dates — this is the key speed/accuracy fix
TOPIC_RANGES = {
    "brexit": ("2012-01-01", END_DATE),
    "covid":  ("2020-01-01", END_DATE),
}

CAP_PER_GROUP = 800   # max articles per (outlet, topic) BEFORE scraping — main speed lever
                      # Guardian/BBC/Daily Mail should all scale close to 1:1 with this —
                      # none of the three are paywalled, unlike Telegraph (dropped, see notes)
MIN_WORDS = 50        # drop scraped BBC/Daily Mail articles shorter than this (failed scrapes / stubs)
SCRAPE_WORKERS = 10

mc_search = mediacloud.api.SearchApi(MC_API_KEY)
mc_directory = mediacloud.api.DirectoryApi(MC_API_KEY)
def year_chunks(start_date, end_date):
    """Split a date range into whole-year chunks. Both fetch_guardian and
    fetch_mc_meta now query and cap PER CHUNK instead of one open-ended query —
    this is what fixes the recency bias we found: an open-ended query capped by
    a total-result cutoff silently favored whichever end of the range the API's
    default sort order returns first (turned out to be "newest"), so raising
    CAP_PER_GROUP just collected more of the same recent slice instead of
    reaching further back in time. Chunking by year makes every year compete
    for its own capped share, regardless of API sort defaults."""
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    chunks = []
    cur = start
    while cur < end:
        nxt = min(datetime(cur.year + 1, 1, 1), end)
        chunks.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return chunks


Paste your Media Cloud API key: ··········
Paste your Guardian Open Platform API key: ··········


### Diagnostic — test the actual GUARDIAN_API_KEY value in this session




In [4]:
test_resp = requests.get(
    "https://content.guardianapis.com/search",
    params={"q": "test", "api-key": GUARDIAN_API_KEY},
    timeout=15,
)
print(f"status: {test_resp.status_code}")
print(test_resp.text[:300])
print(f"key length: {len(GUARDIAN_API_KEY)}, starts with: '{GUARDIAN_API_KEY[:4]}', "
      f"ends with: '{GUARDIAN_API_KEY[-4:]}'")


status: 200
{"response":{"status":"ok","userTier":"developer","total":186908,"startIndex":1,"pageSize":10,"currentPage":1,"pages":18691,"orderBy":"relevance","results":[{"id":"australia-news/2026/may/27/morning-mail-wednesday-ntwnfb","type":"article","sectionId":"australia-news","sectionName":"Australia news","
key length: 36, starts with: 'ccce', ends with: '4cf5'


## Part 1 — Guardian (own API, full text included, no scraping)

In [5]:
def fetch_guardian(keywords, topic, start_date, end_date):
    query = " OR ".join(keywords)
    chunks = year_chunks(start_date, end_date)
    per_chunk_cap = max(100, CAP_PER_GROUP // max(len(chunks), 1) * 3)
    rows = []
    for chunk_start, chunk_end in chunks:
        page, total_pages, chunk_rows = 1, 1, []
        while page <= total_pages and len(chunk_rows) < per_chunk_cap:
            resp = requests.get(
                "https://content.guardianapis.com/search",
                params={
                    "q": query,
                    "from-date": chunk_start,
                    "to-date": chunk_end,
                    "page-size": 200,
                    "page": page,
                    "show-fields": "bodyText",
                    "api-key": GUARDIAN_API_KEY,
                },
                timeout=15,
            )
            if resp.status_code != 200:
                print(f"Guardian API error on page {page} ({chunk_start} to {chunk_end}): "
                      f"{resp.status_code} {resp.text[:200]}")
                break
            data = resp.json()["response"]
            total_pages = data["pages"]
            for item in data["results"]:
                chunk_rows.append({
                    "title": item.get("webTitle"),
                    "url": item.get("webUrl"),
                    "publish_date": item.get("webPublicationDate"),
                    "domain": "theguardian.com",
                    "topic": topic,
                    "outlet": "guardian",
                    "text": item.get("fields", {}).get("bodyText", ""),
                })
            page += 1
            time.sleep(0.2)
        rows.extend(chunk_rows)
    return pd.DataFrame(rows)

guardian_frames = []
for topic, kws in [("brexit", BREXIT_KEYWORDS), ("covid", COVID_KEYWORDS)]:
    start_date, end_date = TOPIC_RANGES[topic]
    df = fetch_guardian(kws, topic, start_date, end_date)
    print(f"Guardian/{topic}: {len(df)} candidates collected ({start_date} to {end_date})")
    guardian_frames.append(df)

guardian_df = pd.concat(guardian_frames, ignore_index=True) if guardian_frames else pd.DataFrame(
    columns=["title", "url", "publish_date", "domain", "topic", "outlet", "text"]
)
guardian_df = guardian_df.drop_duplicates(subset=["url"]).reset_index(drop=True) if not guardian_df.empty else guardian_df
if guardian_df.empty:
    print("Guardian collected 0 articles — check the API errors printed above (likely an "
          "invalid/expired API key, e.g. a 401) before continuing. Stopping here rather than "
          "crashing further down with a confusing error.")
    raise SystemExit("Guardian collection failed — see message above.")
guardian_df["word_count"] = guardian_df["text"].fillna("").apply(lambda t: len(t.split()))
guardian_df = guardian_df[guardian_df["word_count"] >= MIN_WORDS].reset_index(drop=True)

# cap per topic before saving
guardian_capped = []
for topic, group in guardian_df.groupby("topic"):
    n = min(CAP_PER_GROUP, len(group))
    guardian_capped.append(group.sample(n, random_state=42))
guardian_df = pd.concat(guardian_capped, ignore_index=True)
print(f"Guardian total after cap: {len(guardian_df)}")
print(guardian_df["topic"].value_counts())
guardian_df["publish_date"] = pd.to_datetime(guardian_df["publish_date"], errors="coerce", utc=True)
print("Date range check (should span the full TOPIC_RANGES window, not just recent months):")
print(guardian_df.groupby("topic")["publish_date"].agg(["min", "max"]))
guardian_df.to_csv(f"{OUTPUT_DIR}/_checkpoint_guardian.csv", index=False)


Guardian/brexit: 3000 candidates collected (2012-01-01 to 2026-08-09)
Guardian/covid: 2800 candidates collected (2020-01-01 to 2026-08-09)
Guardian total after cap: 1600
topic
brexit    800
covid     800
Name: count, dtype: int64
Date range check (should span the full TOPIC_RANGES window, not just recent months):
                             min                       max
topic                                                     
brexit 2012-02-10 21:00:04+00:00 2026-08-04 04:00:12+00:00
covid  2020-03-23 20:41:20+00:00 2026-08-08 20:03:34+00:00


## Part 2 — BBC & Daily Mail metadata via Media Cloud (topic-specific ranges, capped)


In [8]:
def resolve_source_id(outlet_name: str, domain: str):
    if MANUAL_SOURCE_IDS.get(outlet_name):
        print(f"{outlet_name}: using manually-set source id {MANUAL_SOURCE_IDS[outlet_name]}")
        return MANUAL_SOURCE_IDS[outlet_name]

    # Media Cloud's name search can return partial matches (e.g. "telegraph.co.uk" also
    # matching "lancashiretelegraph.co.uk"), and an exact name match can be a dead/unused
    # stub (stories_per_week=0) while a completely unrelated outlet that merely shares
    # part of the name has real volume (e.g. "dailymail.co.uk" itself shows 0 stories/week
    # in Media Cloud right now, while the unrelated "hulldailymail.co.uk" has 151/week).
    # A "pick the highest-volume candidate" fallback is NOT safe here — it has already
    # picked the wrong outlet twice. So: only auto-resolve on an exact name match that's
    # also active. Anything else stops and asks for a manual decision instead of guessing.
    candidates = [domain]
    if domain.startswith("www."):
        candidates.append(domain[4:])
    else:
        candidates.append("www." + domain)
    if domain.endswith(".co.uk"):
        candidates.append(domain[:-6] + ".com")
    candidates_lower = [c.lower() for c in candidates]

    all_matches = {}
    for name in candidates:
        try:
            results = mc_directory.source_list(name=name)
            sources = results.get("results", []) if isinstance(results, dict) else (results or [])
        except Exception as e:
            print(f"{outlet_name}: lookup failed for '{name}': {e}")
            continue
        for s in sources:
            all_matches[s.get("id")] = s

    if not all_matches:
        print(f"{outlet_name}: no matches found for any of {candidates}. Search "
              f"https://search.mediacloud.org/directory manually and set MANUAL_SOURCE_IDS.")
        return None

    print(f"{outlet_name}: {len(all_matches)} candidate source(s) found:")
    for s in all_matches.values():
        print(f"    id={s.get('id')}, name={s.get('name')}, label={s.get('label')}, "
              f"homepage={s.get('homepage')}, stories_per_week={s.get('stories_per_week')}")

    exact_active = [
        s for s in all_matches.values()
        if s.get("name", "").lower() in candidates_lower and (s.get("stories_per_week") or 0) > 0
    ]
    if exact_active:
        match = max(exact_active, key=lambda s: s.get("stories_per_week") or 0)
        print(f"{outlet_name}: picked EXACT + ACTIVE match -> id={match.get('id')}, "
              f"name={match.get('name')}, label={match.get('label')}")
        return match.get("id")

    print(f"{outlet_name}: no exact+active match — NOT guessing. Look through the candidates "
          f"printed above yourself: check the 'homepage' field for each one to confirm which "
          f"is really {outlet_name}'s main site, even if its stories_per_week looks low or 0 "
          f"(Media Cloud's indexing can be stale for a source that's still a valid match). "
          f"Then set MANUAL_SOURCE_IDS['{outlet_name}'] = <that id> in the config cell and "
          f"rerun from there. If nothing here looks right, search "
          f"https://search.mediacloud.org/directory directly instead.")
    return None

SOURCE_IDS = {name: resolve_source_id(name, domain) for name, domain in MC_OUTLETS.items()}
print(SOURCE_IDS)


bbc: 16 candidate source(s) found:
    id=932549, name=bbc.com, label=BBC, homepage=http://bbc.com, stories_per_week=4493
    id=892290, name=www.bbc.co.uk, label=www-bbc-co-uk.cdn.ampproject.org, homepage=http://www-bbc-co-uk.cdn.ampproject.org/, stories_per_week=0
    id=18204, name=bbc.com, label=‭BBC Arabic‬ - ‮الرئيسية‬, homepage=http://www.bbc.com/arabic/, stories_per_week=168
    id=693999, name=heritagebbc.com, label=heritagebbc.com, homepage=http://heritagebbc.com/, stories_per_week=0
    id=1444586, name=football-bbc.com, label=football-bbc.com, homepage=http://football-bbc.com/, stories_per_week=0
    id=1507614, name=magallanesbbc.com.ve, label=magallanesbbc.com.ve, homepage=http://magallanesbbc.com.ve/, stories_per_week=0
    id=864792, name=edgesuite-staging.net, label=bbc.com.edgesuite-staging.net, homepage=http://bbc.com.edgesuite-staging.net/, stories_per_week=0
    id=1514225, name=wvbbc.com, label=wvbbc.com:443, homepage=http://wvbbc.com:443/, stories_per_week=None
 

In [9]:
def fetch_mc_meta(outlet_name, source_id, keywords, topic, start_date, end_date):
    query = " OR ".join(f'"{k}"' for k in keywords)
    chunks = year_chunks(start_date, end_date)
    per_chunk_cap = max(100, CAP_PER_GROUP // max(len(chunks), 1) * 3)
    rows = []
    for chunk_start, chunk_end in chunks:
        chunk_rows, pagination_token, more = [], None, True
        while more and len(chunk_rows) < per_chunk_cap:
            try:
                page, pagination_token = mc_search.story_list(
                    query,
                    start_date=datetime.strptime(chunk_start, "%Y-%m-%d").date(),
                    end_date=datetime.strptime(chunk_end, "%Y-%m-%d").date(),
                    source_ids=[source_id],
                    pagination_token=pagination_token,
                )
            except Exception as e:
                print(f"MC failed: {outlet_name}/{topic} ({chunk_start} to {chunk_end}): {e}")
                break
            for s in page:
                chunk_rows.append({
                    "title": s.get("title"), "url": s.get("url"),
                    "publish_date": s.get("publish_date"),
                    "domain": MC_OUTLETS[outlet_name], "topic": topic, "outlet": outlet_name,
                })
            more = pagination_token is not None
            time.sleep(0.3)
        rows.extend(chunk_rows)
    return pd.DataFrame(rows)

mc_frames = []
for outlet_name, source_id in SOURCE_IDS.items():
    if not source_id:
        print(f"Skipping {outlet_name} — no source id resolved")
        continue
    for topic, kws in [("brexit", BREXIT_KEYWORDS), ("covid", COVID_KEYWORDS)]:
        start_date, end_date = TOPIC_RANGES[topic]
        df = fetch_mc_meta(outlet_name, source_id, kws, topic, start_date, end_date)
        print(f"MC {outlet_name}/{topic}: {len(df)} candidates ({start_date} to {end_date})")
        mc_frames.append(df)

mc_meta = pd.concat(mc_frames, ignore_index=True) if mc_frames else pd.DataFrame(
    columns=["title", "url", "publish_date", "domain", "topic", "outlet"]
)
if mc_meta.empty:
    print("No BBC/Daily Mail metadata collected — check the source id resolution output "
          "above (previous cell) before continuing. Guardian will still work on its own.")
mc_meta["publish_date"] = pd.to_datetime(mc_meta["publish_date"], errors="coerce", utc=True)
mc_meta = mc_meta.dropna(subset=["url"]).drop_duplicates(subset=["url"]).reset_index(drop=True)

# cap per (outlet, topic) BEFORE scraping — this is what keeps runtime short
mc_meta_capped = []
for (outlet_name, topic), group in mc_meta.groupby(["outlet", "topic"]):
    n = min(CAP_PER_GROUP, len(group))
    mc_meta_capped.append(group.sample(n, random_state=42))
mc_meta = pd.concat(mc_meta_capped, ignore_index=True) if mc_meta_capped else mc_meta

print("After capping, going into scraping:")
print(mc_meta.groupby(["outlet", "topic"]).size())
print("Date range check (should span the full TOPIC_RANGES window, not just recent months):")
print(mc_meta.groupby(["outlet", "topic"])["publish_date"].agg(["min", "max"]))
mc_meta.to_csv(f"{OUTPUT_DIR}/_checkpoint_mc_meta.csv", index=False)


MC bbc/brexit: 11076 candidates (2012-01-01 to 2026-08-09)
MC bbc/covid: 7000 candidates (2020-01-01 to 2026-08-09)
MC dailymail/brexit: 13600 candidates (2012-01-01 to 2026-08-09)
MC dailymail/covid: 7000 candidates (2020-01-01 to 2026-08-09)
After capping, going into scraping:
outlet     topic 
bbc        brexit    800
           covid     800
dailymail  brexit    800
           covid     800
dtype: int64
Date range check (should span the full TOPIC_RANGES window, not just recent months):
                                       min                       max
outlet    topic                                                     
bbc       brexit 2012-02-06 00:00:00+00:00 2026-07-21 00:00:00+00:00
          covid  2020-01-10 00:00:00+00:00 2026-08-08 00:00:00+00:00
dailymail brexit 2012-01-28 00:00:00+00:00 2026-03-16 00:00:00+00:00
          covid  2020-01-05 00:00:00+00:00 2026-03-16 00:00:00+00:00


## Part 3 — scrape full text for BBC & Daily Mail

Only `CAP_PER_GROUP`-sized samples now, so this should take a couple of minutes. **Telegraph was dropped as an outlet** — its paywall blocked essentially all scraping (0 articles survived even after correctly resolving its real source id; see the diagnostic cell below, kept for reference). Daily Mail is free and should scrape cleanly.


In [10]:
HEADERS = {"User-Agent": "Mozilla/5.0 (research script; masters thesis data collection)"}

def scrape_text(url, timeout=10):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        if resp.status_code != 200:
            return None
        return trafilatura.extract(resp.text)
    except Exception:
        return None

def scrape_all(urls, workers=SCRAPE_WORKERS):
    results = {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(scrape_text, u): u for u in urls}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            url = futures[fut]
            results[url] = fut.result()
    return results

mc_outlet_dfs = {}
for outlet_name in MC_OUTLETS:
    meta = mc_meta[mc_meta["outlet"] == outlet_name].copy()
    print(f"Scraping {len(meta)} {outlet_name} URLs...")
    text_map = scrape_all(meta["url"].tolist())
    meta["text"] = meta["url"].map(text_map)
    meta["word_count"] = meta["text"].fillna("").apply(lambda t: len(t.split()))
    before = len(meta)
    meta = meta[meta["word_count"] >= MIN_WORDS].reset_index(drop=True)
    print(f"{outlet_name}: {before} scraped -> {len(meta)} kept after MIN_WORDS filter")
    mc_outlet_dfs[outlet_name] = meta
    meta.to_csv(f"{OUTPUT_DIR}/_checkpoint_{outlet_name}_scraped.csv", index=False)

Scraping 1600 bbc URLs...


100%|██████████| 1600/1600 [02:51<00:00,  9.33it/s]


bbc: 1600 scraped -> 1584 kept after MIN_WORDS filter
Scraping 1600 dailymail URLs...


100%|██████████| 1600/1600 [05:14<00:00,  5.09it/s]


dailymail: 1600 scraped -> 1597 kept after MIN_WORDS filter


### Optional diagnostic — inspect what a scrape actually returns



In [11]:
import requests, trafilatura

sample_urls = mc_meta[mc_meta["outlet"] == "dailymail"]["url"].head(3).tolist()
print(f"Testing {len(sample_urls)} Daily Mail URLs (optional diagnostic, not expected to be needed):\n")

for url in sample_urls:
    print(f"--- {url} ---")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        print(f"status_code: {resp.status_code}")
        print(f"response length: {len(resp.text)} chars")
        print(f"first 300 chars of raw HTML:\n{resp.text[:300]}")
        extracted = trafilatura.extract(resp.text)
        print(f"trafilatura extracted: {len(extracted) if extracted else 0} chars")
        if extracted:
            print(f"extracted preview: {extracted[:200]}")
    except Exception as e:
        print(f"request failed: {e}")
    print()


Testing 3 Daily Mail URLs (optional diagnostic, not expected to be needed):

--- http://www.dailymail.co.uk/money/markets/article-2370492/Vodafone-hit-triple-whammy-European-businesses.html?ITO=1490&ns_mchannel=rss&ns_campaign=1490 ---
status_code: 200
response length: 759174 chars
first 300 chars of raw HTML:
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "//www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd"><html class="article-page"><head>
    
        
    
<script type="text/javascript">
try {
  Object.defineProperty(window, 'adverts', {configurable: false, value:{}});
}
catch(error) {
trafilatura extracted: 2739 chars
extracted preview: Vodafone hit by 'triple whammy' across its European businesses in last three months
- See more This is Money on Google - save us as a Preferred Source
Vodafone was hit by a ‘triple whammy’ across its 

--- http://www.dailymail.co.uk/news/article-3453809/Let-migrants-demand-luvvies-String-millionaire-celebrities-sign-letter-urging-Ca

## Part 4 — save final per-outlet datasets (no further balancing needed — already capped)

In [12]:
all_outlet_dfs = dict(mc_outlet_dfs)
all_outlet_dfs["guardian"] = guardian_df

for outlet_name, df in all_outlet_dfs.items():
    df = df.copy()
    df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce", utc=True)
    df = df.sort_values("publish_date").reset_index(drop=True)
    filename = f"{OUTPUT_DIR}/{outlet_name}_brexit_covid_text_FINAL.csv"
    df[["title", "url", "publish_date", "domain", "topic", "text", "word_count"]].to_csv(filename, index=False)
    print(f"{outlet_name}: saved {len(df)} rows -> {filename}")
    print(df["topic"].value_counts())
    print()

print("Done. Final CSVs are in your Google Drive under:", OUTPUT_DIR)

bbc: saved 1584 rows -> /content/drive/MyDrive/brexit_covid_datasets/bbc_brexit_covid_text_FINAL.csv
topic
brexit    794
covid     790
Name: count, dtype: int64

dailymail: saved 1597 rows -> /content/drive/MyDrive/brexit_covid_datasets/dailymail_brexit_covid_text_FINAL.csv
topic
brexit    799
covid     798
Name: count, dtype: int64

guardian: saved 1600 rows -> /content/drive/MyDrive/brexit_covid_datasets/guardian_brexit_covid_text_FINAL.csv
topic
brexit    800
covid     800
Name: count, dtype: int64

Done. Final CSVs are in your Google Drive under: /content/drive/MyDrive/brexit_covid_datasets
